# 「誰が・何を・どうした」の三つ組を抜き出す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_parsing/svo_extract.ipynb)

NLP クックブック 18-09 ／ 構文解析

文から 主語-目的語-述語 の組を取り出して、簡単な情報抽出や知識グラフの素材にしたい。

In [1]:
!pip install -q ginza ja_ginza

In [2]:
import ginza
import spacy

nlp = spacy.load("ja_ginza")

def phrase(token):
    """トークンを含む文節から、後ろに付く助詞などを除いた部分を返す"""
    return "".join(t.text for t in ginza.bunsetu_span(token) if t.i <= token.i)

def triples(text):
    out = []
    for sent in nlp(text).sents:
        for verb in sent:
            if verb.pos_ not in ("VERB", "ADJ"):
                continue
            args = {c.dep_: phrase(c) for c in verb.children if c.dep_ in ("nsubj", "obj", "iobj", "obl")}
            if "nsubj" in args:
                out.append((args.get("nsubj"), args.get("obj") or args.get("obl"), verb.lemma_))
    return out

for s in ["OpenAI が新しい言語モデルを発表した。", "山田教授は学生に自然言語処理を教えている。",
          "猫がネズミを追いかけ、犬が猫を追いかけた。", "政府は来年度の予算案を閣議で決定した。"]:
    print(s, "→", triples(s))

OpenAI が新しい言語モデルを発表した。 → [('OpenAI', '言語モデル', '発表')]
山田教授は学生に自然言語処理を教えている。 → [('山田教授', '自然言語処理', '教える')]
猫がネズミを追いかけ、犬が猫を追いかけた。 → [('猫', 'ネズミ', '追いかける'), ('犬', '猫', '追いかける')]
政府は来年度の予算案を閣議で決定した。 → [('政府', '予算案', '決定')]


- 日本語は主語の省略が多く、「は」で示される主題が必ずしも主語ではない。本格的には述語項構造解析やゼロ照応解析が必要。
- 受身・使役では格と意味役割がずれる（「ネズミが猫に追いかけられた」）。